# 01 — Exploratory Data Analysis & Baseline Model

This notebook performs the first pass of analysis for the Credit Risk Intelligence System.

Goals:

1. Load the Home Credit training dataset
2. Understand target imbalance
3. Inspect missing values and feature types
4. Build a baseline Logistic Regression model
5. Evaluate the model using credit-risk appropriate metrics

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [4]:
df = pd.read_csv("../data/raw/application_train.csv")

df.shape

(307511, 122)

In [6]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
df["TARGET"].value_counts()

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [10]:
df["TARGET"].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [12]:
df.dtypes.value_counts()

float64    65
int64      41
object     16
Name: count, dtype: int64

In [14]:
target = "TARGET"

X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

len(numeric_features), len(categorical_features)

(105, 16)

In [19]:
missing_summary = (
    df.isnull()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["column", "missing_rate"]

missing_summary.head(25)

,column,missing_rate
0,COMMONAREA_MEDI,0.698723
1,COMMONAREA_AVG,0.698723
2,COMMONAREA_MODE,0.698723
3,NONLIVINGAPARTMENTS_MODE,0.694330
4,NONLIVINGAPARTMENTS_AVG,0.694330
5,NONLIVINGAPARTMENTS_MEDI,0.694330
6,FONDKAPREMONT_MODE,0.683862
7,LIVINGAPARTMENTS_MODE,0.683550
8,LIVINGAPARTMENTS_AVG,0.683550
9,LIVINGAPARTMENTS_MEDI,0.683550


In [21]:
high_missing_cols = missing_summary[missing_summary["missing_rate"] > 0.40]["column"].tolist()

len(high_missing_cols), high_missing_cols[:10]

(49,
 ['COMMONAREA_MEDI',
  'COMMONAREA_AVG',
  'COMMONAREA_MODE',
  'NONLIVINGAPARTMENTS_MODE',
  'NONLIVINGAPARTMENTS_AVG',
  'NONLIVINGAPARTMENTS_MEDI',
  'FONDKAPREMONT_MODE',
  'LIVINGAPARTMENTS_MODE',
  'LIVINGAPARTMENTS_AVG',
  'LIVINGAPARTMENTS_MEDI'])

In [23]:
target = "TARGET"
id_column = "SK_ID_CURR"

columns_to_drop = [id_column] + high_missing_cols

X = df.drop(columns=[target] + columns_to_drop)
y = df[target]

X.shape

(307511, 71)

In [25]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features before encoding:", X.shape[1])

Numeric features: 59
Categorical features: 12
Total features before encoding: 71


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training target rate:")
print(y_train.value_counts(normalize=True))
print("Testing target rate:")
print(y_test.value_counts(normalize=True))

Training rows: 246008
Testing rows: 61503
Training target rate:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64
Testing target rate:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [29]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [31]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
y_pred = logistic_model.predict(X_test)
y_pred_proba = logistic_model.predict_proba(X_test)[:, 1]

In [35]:
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("ROC-AUC:", round(roc_auc, 4))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

ROC-AUC: 0.7453

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.69      0.80     56538
           1       0.16      0.67      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.68      0.53     61503
weighted avg       0.90      0.69      0.76     61503



In [37]:
confusion_matrix(y_test, y_pred)

array([[38971, 17567],
       [ 1616,  3349]])

In [39]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Non-Default", "Actual Default"],
    columns=["Predicted Non-Default", "Predicted Default"]
)

cm_df

,Predicted Non-Default,Predicted Default
Actual Non-Default,38971,17567
Actual Default,1616,3349


## Baseline Model Interpretation

The Logistic Regression model is used as the first baseline because it is simple, interpretable, and useful for comparing more advanced models later.

Because the dataset is highly imbalanced, accuracy is not the main success metric. A model could achieve high accuracy by predicting most applicants as non-default.

For this project, ROC-AUC, recall for the default class, precision, and the confusion matrix are more important.